In [2]:
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")

# خواندن همه فایل‌های parquet در مسیر مشخص
dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/022026/Data/MEDS_MDS/data/held_out/*.parquet')]    #     Zahra/Data-07-2025/MDP/MEDS_811/data/train
)

# تبدیل به pandas DataFrame
df = dataset.to_pandas_dataframe()
df.head(15)


{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,subject_id,time,code,numeric_value
0,5,1992-03-12 00:00:00,DOB,NaN
1,5,2017-04-16 00:00:00,D/DR040,NaN
2,5,2017-04-16 21:43:00,ADMISSION_ADT,NaN
3,5,2017-04-16 21:43:00,MOVE_ADT,NaN
4,5,2017-04-16 21:43:00,^AFSNIT_ADT/,NaN
5,5,2017-04-16 23:07:00,ADMISSION_ADT,NaN
6,5,2017-04-16 23:07:00,MOVE_ADT,NaN
7,5,2017-04-16 23:07:00,^AFSNIT_ADT/,NaN
8,5,2017-04-17 07:35:00,ADMISSION_ADT,NaN
9,5,2017-04-17 07:35:00,MOVE_ADT,NaN


In [3]:
len(df)

45468522

In [4]:
import pandas as pd

# فرض بر این که فایل CSV رو داری
# df = pd.read_csv("your_file.csv")  # یا مستقیم اگر DataFrame آماده‌ست، نیازی نیست

# دسته‌بندی هر subject_id بر اساس وجود کدهای مختلف
m_patients = df[df['code'].str.startswith('M/', na=False)]['subject_id'].unique()
p_patients = df[df['code'].str.startswith('P/', na=False)]['subject_id'].unique()
d_patients = df[df['code'].str.startswith('D/', na=False)]['subject_id'].unique()
s_patients = df[df['code'].str.startswith('S/', na=False)]['subject_id'].unique()

# کل بیماران منحصربه‌فرد
all_patients = df['subject_id'].unique()

# نمایش آمار
print(f"Whole Patients: {len(all_patients)}")
print(f"The patients has M-medication Codes: {len(m_patients)}")
print(f"The patients has D-diagnosis Codes: {len(d_patients)}")
print(f"The patients has P-Procedure Codes: {len(p_patients)}")
print(f"The patients has S-SKS Codes: {len(s_patients)}")


Whole Patients: 221803
The patients has M-medication Codes: 151356
The patients has D-diagnosis Codes: 221749
The patients has P-Procedure Codes: 0
The patients has S-SKS Codes: 53203


In [5]:
subject_counts = df['subject_id'].value_counts()


In [6]:
subject_counts

106        60297
1154761    47724
1580453    43790
1011632    43454
926005     40467
           ...  
405889         2
180101         2
1755398        2
816167         2
1629822        2
Name: subject_id, Length: 221803, dtype: int64

In [7]:
p_Num = df[df['code'].str.startswith('S/', na=False)]

In [8]:
p_Num

,subject_id,time,code,numeric_value
74,5,2020-01-26 23:59:00,S/KEBA10,NaN
117,5,2020-10-19 23:59:00,S/KNFH91B,NaN
247,70,2017-01-11 23:59:00,S/KNCJ65,NaN
266,70,2017-08-20 23:59:00,S/KNBL49,NaN
299,70,2018-05-27 23:59:00,S/KNDU49,NaN
...,...,...,...,...
45465740,2217214,2019-07-22 23:59:00,S/KKAC00,NaN
45466008,2217394,2020-11-18 23:59:00,S/KNGD21,NaN
45466381,2217809,2022-03-23 23:59:00,S/KJFA15,NaN
45466477,2217869,2018-10-18 23:59:00,S/KFNG05,NaN


In [9]:
import pandas as pd

# پیدا کردن سطرهایی که فقط codeهای نوع /P دارن
only_p = df[df['code'].str.startswith('S/', na=False)]

# بیماران با فقط /P کد
subject_ids_only_p = only_p['subject_id'].unique()

# حالا بیماران با codeهای غیر از /P
not_p = df[~df['code'].str.startswith('P/', na=False)]
subject_ids_with_non_p = set(not_p['subject_id'].unique())

# حذف بیمارانی که فقط /P دارن
only_p_ids_to_exclude = [sid for sid in subject_ids_only_p if sid not in subject_ids_with_non_p]

print("Number of patients with only Surgery code: ", only_p_ids_to_exclude)


Number of patients with only Surgery code:  []


In [11]:
df_filtered = df[~df['code'].str.startswith('S/', na=False)]

In [15]:
subject_counts_MD = df_filtered['subject_id'].value_counts()

In [17]:
subject_counts_MD

106        60295
1154761    47724
1580453    43789
1011632    43454
926005     40467
           ...  
1629822        2
528769         2
1044530        2
190688         2
1335726        2
Name: subject_id, Length: 221803, dtype: int64

In [19]:
subject_counts_df = subject_counts.reset_index()
subject_counts_df.columns = ['subject_id', 'original_count']

subject_counts_MD_df = subject_counts_MD.reset_index()
subject_counts_MD_df.columns = ['subject_id', 'new_count']


In [20]:
import pandas as pd
comparison_df = pd.merge(subject_counts_df, subject_counts_MD_df, on='subject_id', how='outer')


In [21]:
comparison_df['difference'] =  comparison_df['original_count'] - comparison_df['new_count']


In [22]:
comparison_df = comparison_df.sort_values(by='difference', ascending=False)


In [23]:
comparison_df

,subject_id,original_count,new_count,difference
7233,1848318,1269,1250,19
7679,2194446,1213,1196,17
918,1528494,4042,4029,13
2359,1087400,2576,2563,13
17478,2079587,589,576,13
...,...,...,...,...
102529,1634170,52,52,0
102530,622047,52,52,0
102531,2078315,52,52,0
102532,1038159,52,52,0


In [24]:
unchanged_count = (comparison_df['difference'] == 0).sum()
print("unchanged_count", unchanged_count)


unchanged_count 168600


In [25]:
comparison_df['abs_diff'] = comparison_df['difference'].abs()
most_changed = comparison_df.sort_values(by='abs_diff', ascending=False)


In [26]:
print(most_changed.head(10))


       subject_id  original_count  new_count  difference  abs_diff
7233      1848318            1269       1250          19        19
7679      2194446            1213       1196          17        17
918       1528494            4042       4029          13        13
2359      1087400            2576       2563          13        13
17478     2079587             589        576          13        13
2664      2004326            2403       2391          12        12
9836      2005354            1006        994          12        12
7257      1031683            1266       1254          12        12
38139     1109842             229        217          12        12
2925      2088712            2290       2279          11        11


In [27]:
changed_df = comparison_df[comparison_df['difference'] != 0]
min_new_count = changed_df['new_count'].min()
lowest_new_count_patients = changed_df[changed_df['new_count'] == min_new_count]


In [28]:
lowest_new_count_patients = lowest_new_count_patients.rename(
    columns={
        'original_count': 'MDS codes',
        'new_count': 'MD codes'
    }
)


In [29]:
lowest_new_count_patients

,subject_id,MDS codes,MD codes,difference,abs_diff
213006,728967,4,3,1,1
213939,613021,4,3,1,1
214085,964284,4,3,1,1


.str.upper() شرط را case-insensitive می‌کند

بل از فیلتر، ستون را به pd.StringDtype() تبدیل می‌کند؛ این کار رفتار .str را پایدار و قابل پیش‌بینی می‌کند (<NA> به‌جای NaN)



In [30]:
# کل بیماران منحصربه‌فرد
all_patients = df_filtered['subject_id'].unique()
len(all_patients)
# نمایش آمار

221803

In [31]:
len(df_filtered)

45391200

In [32]:
df_filtered

,subject_id,time,code,numeric_value
0,5,1992-03-12 00:00:00,DOB,NaN
1,5,2017-04-16 00:00:00,D/DR040,NaN
2,5,2017-04-16 21:43:00,ADMISSION_ADT,NaN
3,5,2017-04-16 21:43:00,MOVE_ADT,NaN
4,5,2017-04-16 21:43:00,^AFSNIT_ADT/,NaN
...,...,...,...,...
45468517,2217869,2024-04-03 13:00:00,M/M05BA08,NaN
45468518,2217869,2024-04-03 13:15:00,M/M05BA08,NaN
45468519,2217869,2024-05-08 11:53:00,M/M05BA08,NaN
45468520,2217869,2024-05-08 12:10:00,M/M05BA08,NaN


In [33]:
import pandas as pd
import numpy as np

# امن‌تر: اگر code نال یا غیررشته‌ای بود اذیت نکنه
df['code'] = df['code'].astype('string')
df_filtered = df[~df['code'].str.upper().str.startswith('S/', na=False)].copy()
print("kept rows MD:", len(df_filtered), " / total:", len(df))


kept rows MD: 45391200  / total: 45468522


In [34]:
import pandas as pd
import numpy as np

# اطمینان از نوع‌ها (برای خروجی تمیز و بدون خطا)
df_filtered = df_filtered.copy()
df_filtered['subject_id']    = pd.to_numeric(df_filtered['subject_id'], errors='coerce').astype('Int64')
df_filtered['numeric_value'] = pd.to_numeric(df_filtered['numeric_value'], errors='coerce').astype('float32')
df_filtered['time']          = pd.to_datetime(df_filtered['time'], errors='coerce', utc=False)

# ردیف‌های بدون subject_id را حذف کنیم (نمی‌توان شارد کرد)
df_filtered = df_filtered.dropna(subset=['subject_id']).copy()
df_filtered['subject_id'] = df_filtered['subject_id'].astype('int64')

# ۳۶ شارد: هر بیمار فقط در یک فایل (mod 36)
#N_SHARDS = 45
#df_filtered['__shard__'] = (df_filtered['subject_id'] % N_SHARDS).astype('int16')

print("rows to write:", len(df_filtered))


rows to write: 45391200


In [35]:
import numpy as np
import os

N_SHARDS = 5   #45 for Whole # 36 when we have split
OUT_DIR = "./_held_outMDP_withoutS_sharded"
os.makedirs(OUT_DIR, exist_ok=True)

cols_out = ['subject_id', 'time', 'code', 'numeric_value']

# فرض: subject_id قبلاً int64 شده و NaNها حذف شده‌اند (طبق سلول قبلی‌ات)
sid_mod = (df_filtered['subject_id'].to_numpy(dtype=np.int64, copy=False) % N_SHARDS)

written = 0
for k in range(N_SHARDS):
    mask = (sid_mod == k)                 # بدون ستون اضافی، فقط یک آرایهٔ NumPy
    part = df_filtered.loc[mask, cols_out].sort_values(['subject_id','time'])
    # اگر می‌خوای حتماً ۳۶ فایل 0..35 داشته باشی حتی اگه خالی باشن:
    # if part.empty:
    #     part = part.iloc[0:0]  # فایل صفر-سطر با همان ستون‌ها
    part.to_parquet(os.path.join(OUT_DIR, f"{k}.parquet"),
                    engine="pyarrow", compression="snappy", index=False)
    written += 1

print(f"Done. wrote {written} parquet files into {OUT_DIR}")


Done. wrote 5 parquet files into ./_held_outMDP_withoutS_sharded


In [36]:
import pyarrow.parquet as pq

OUT_DIR = "./_held_outMDP_withoutS_sharded"

def parquet_num_rows(path):
    pf = pq.ParquetFile(path)
    md = pf.metadata
    return sum(md.row_group(i).num_rows for i in range(md.num_row_groups))

total = 0
for f in sorted(p for p in os.listdir(OUT_DIR) if p.endswith('.parquet')):
    n = parquet_num_rows(os.path.join(OUT_DIR, f))
    total += n
    print(f, "rows:", n)
print("TOTAL rows:", total)


0.parquet rows: 9348777
1.parquet rows: 9084291
2.parquet rows: 9065188
3.parquet rows: 9126315
4.parquet rows: 8766629
TOTAL rows: 45391200


In [37]:
from azureml.data.datapath import DataPath
from azureml.data.dataset_factory import FileDatasetFactory
import os

OUT_DIR = "./_held_outMDP_withoutS_sharded"
DST_PREFIX = "Zahra/022026/Data/MEDS_MD/data/held_out"  #held_out"  # بدون اسلشِ اول
''

# اطمینان: پوشه خروجی وجود دارد و فایل parquet داخلش هست
print("Local files to upload:", len([f for f in os.listdir(OUT_DIR) if f.endswith(".parquet")]))

# مقصد روی Datastore
target = DataPath(datastore, DST_PREFIX)

# آپلود همه محتویات OUT_DIR به DST_PREFIX
_ = FileDatasetFactory.upload_directory(
    src_dir=OUT_DIR,
    target=target,
    overwrite=True,
    show_progress=True,
)
print(f"Uploaded to datastore path: {DST_PREFIX}")


Local files to upload: 5
Validating arguments.
Arguments validated.
'overwrite' is set to True. Any file already present in the target will be overwritten.
Uploading files from '/mnt/batch/tasks/shared/LS_root/mounts/clusters/zahracpu/code/Users/zahra.sobhaninia/Corebehrt_CV/CreateIdenticalData/_held_outMDP_withoutS_sharded' to 'Zahra/022026/Data/MEDS_MD/data/held_out'
Copying 5 files with concurrency set to 4
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/zahracpu/code/Users/zahra.sobhaninia/Corebehrt_CV/CreateIdenticalData/_held_outMDP_withoutS_sharded/1.parquet, file 1 out of 5. Destination path: https://forskerpln0ybkrdls01.dfs.core.windows.net/researcher-data/Zahra/022026/Data/MEDS_MD/data/held_out/1.parquet
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/zahracpu/code/Users/zahra.sobhaninia/Corebehrt_CV/CreateIdenticalData/_held_outMDP_withoutS_sharded/2.parquet, file 2 out of 5. Destination path: https://forskerpln0ybkrdls01.dfs.core.windows.net/researcher-data/Za

In [38]:
paths = Dataset.File.from_files(path=[(datastore, f"{DST_PREFIX}/*.parquet")]).to_path()
print("Found in datastore:", len(paths))
print(paths[:20])


{'infer_column_types': 'False', 'activity': 'to_path'}
{'infer_column_types': 'False', 'activity': 'to_path', 'activityApp': 'FileDataset'}
Found in datastore: 5
['/0.parquet', '/1.parquet', '/2.parquet', '/3.parquet', '/4.parquet']
